In [1]:
import sys

import numpy as np
import polars as pl
from aeon.datasets import load_classification
from sklearn.preprocessing import StandardScaler

sys.path.append("/app")

from src.polars_utils import random_split


def process_uae_split(ds_name, split, scaler: StandardScaler | None = None):
    """Process a UAE dataset split."""
    X, y = load_classification(ds_name, extract_path="/mnt/data/raw", split=split)
    N, F, L = X.shape
    X = X.transpose(0, 2, 1).reshape(N * L, F)
    if split == "train":
        scaler = StandardScaler().fit(X)
    X = scaler.transform(X).reshape(N, L, F)
    M = int(L * 0.3)

    rng = np.random.default_rng(42)
    X[np.arange(N)[:, None], rng.choice(L, (N, M))] = np.nan
    time = np.broadcast_to(np.linspace(0, 1, L), (N, L))

    df_dict = {"target": y.tolist(), "time": time.tolist()}
    for i in range(F):
        df_dict[f"feature_{i}"] = X[:, :, i].tolist()

    df = pl.DataFrame(df_dict)
    return df, scaler


def process_uae_ds(load_name: str, save_name: str):
    """Process a UAE dataset."""
    train_df, scaler = process_uae_split(load_name, "train")
    train_df = random_split(train_df, train=0.8, val=0.2)

    test_df = process_uae_split(load_name, "test", scaler=scaler)[0]
    test_df = test_df.with_columns(split=pl.lit("test"))
    df = pl.concat([train_df, test_df]).with_columns(
        pl.col("target").cast(pl.Categorical).to_physical()
    )

    df.write_parquet(f"/mnt/data/preprocessed/{save_name}.parquet")

In [ ]:
process_uae_ds("UWaveGestureLibrary", "uwgl")
process_uae_ds("InsectSound", "is")